## Step 1: Import Libraries and Configuration


In [ ]:
# Import essential libraries and packages for the analysis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, RobustScaler
from sklearn.model_selection import (
    train_test_split, cross_val_score, GridSearchCV,
    RandomizedSearchCV, learning_curve
)
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_regression
import warnings

warnings.filterwarnings('ignore')

# Analysis tools
from scipy import stats
from scipy.stats import pearsonr, spearmanr
import itertools

# Add imports for file downloading
import requests
import os
from pathlib import Path

print("✓ All libraries imported successfully!")


In [ ]:
# Configuration for professional presentation
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.figsize'] = (12, 8)
sns.set_style("whitegrid")
sns.set_palette("husl")

# Set personalized random seed
STUDENT_SEED = 18  # Example: Replace with your actual student ID last 2 digits
np.random.seed(STUDENT_SEED)

print(f"✓ Configuration set successfully!")
print(f"Random seed: {STUDENT_SEED}")


# PART 1: ENERGY EFFICIENCY ANALYSIS


## Step 2: Energy Dataset - Download Function


In [ ]:
def download_file(url, local_filename):
    """Download file from URL to local directory if it doesn't exist"""
    if not os.path.exists(local_filename):
        print(f"Downloading {local_filename} from {url}...")
        try:
            response = requests.get(url, stream=True)
            response.raise_for_status()

            with open(local_filename, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"✓ File downloaded successfully: {local_filename}")
            return True
        except Exception as e:
            print(f"✗ Error downloading file: {e}")
            return False
    else:
        print(f"✓ File already exists locally: {local_filename}")
        return True


print("✓ Download function defined")


## Step 3: Energy Dataset - Load Data


In [ ]:
print("=" * 60)
print("PART 1: ENERGY EFFICIENCY ANALYSIS")
print("=" * 60)

# Load Energy Efficiency Dataset with download functionality
print("Loading Energy Efficiency Dataset...")

# Define the URL and local file path
energy_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00242/ENB2012_data.xlsx'
energy_local_file = 'ENB2012_data.xlsx'

# Try to download and read the energy dataset
try:
    download_success = download_file(energy_url, energy_local_file)

    if download_success and os.path.exists(energy_local_file):
        energy_df = pd.read_excel(energy_local_file)
        print(f"✓ Dataset loaded from local file! Shape: {energy_df.shape}")
    else:
        print("Trying direct URL reading as fallback...")
        energy_df = pd.read_excel(energy_url)
        print(f"✓ Dataset loaded from URL! Shape: {energy_df.shape}")

except Exception as e:
    print(f"✗ Error loading dataset: {e}")
    print("Please ensure you have internet connection or the file is available locally")


## Step 4: Energy Dataset - Basic Information


In [ ]:
# Rename columns to descriptive names
energy_df.rename(columns={
    'X1': 'Relative_Compactness',
    'X2': 'Surface_Area',
    'X3': 'Wall_Area',
    'X4': 'Roof_Area',
    'X5': 'Overall_Height',
    'X6': 'Orientation',
    'X7': 'Glazing_Area',
    'X8': 'Glazing_Area_Distribution',
    'Y1': 'Heating_Load',
    'Y2': 'Cooling_Load'
}, inplace=True)

print("Dataset Info:")
print(energy_df.info())
print("\nFirst 5 rows:")
print(energy_df.head())


## Step 5: Energy Dataset - Target Correlation Analysis


In [ ]:
"""
UNDERSTANDING P-VALUE - A DETAILED EXPLANATION:

The p-value is one of the most misunderstood concepts in statistics. Let's break it down:

1. CORRELATION COEFFICIENT (r):
   - Measures LINEAR relationship strength between two variables
   - Range: -1 to +1
   - r = +1: Perfect positive correlation
   - r = 0:  No linear relationship
   - r = -1: Perfect negative correlation
   - |r| > 0.7: Strong correlation
   - 0.3 < |r| < 0.7: Moderate correlation
   - |r| < 0.3: Weak correlation

2. P-VALUE:
   - Probability of observing this correlation (or stronger) by random chance
   - Tests null hypothesis: "No correlation exists (r = 0)"
   - p < 0.05: Statistically significant (reject null hypothesis)
   - p < 0.001: Highly significant
   - p = 0.0000: Extremely significant (actual value < 0.0001)
"""

# Analyze the dual-target nature: Compute correlation between Heating_Load and Cooling_Load
corr, p_value = pearsonr(energy_df['Heating_Load'], energy_df['Cooling_Load'])

print("=" * 80)
print("🔍 DETAILED P-VALUE EXPLANATION WITH OUR DATA")
print("=" * 80)

print(f"📊 Correlation Coefficient (r): {corr:.4f}")
print(f"🎯 P-value: {p_value:.10f}")

print(f"\n🤔 WHAT DOES P-VALUE = {p_value:.10f} MEAN?")
print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

if p_value < 0.0001:
    print(f"🎲 LOTTERY ANALOGY:")
    print(f"   Imagine buying a lottery ticket with 1 in 10,000+ chance of winning")
    print(f"   P-value ≈ 0.0000 means finding this correlation by random chance")
    print(f"   is like winning that lottery - technically possible but extremely unlikely!")

    print(f"\n🔬 SCIENTIFIC INTERPRETATION:")
    print(f"   If we repeated this experiment 10,000 times with truly unrelated variables,")
    print(f"   we'd expect to see correlation ≥ {corr:.4f} in fewer than 1 trial.")

    print(f"\n💡 PLAIN ENGLISH:")
    print(f"   'There's virtually ZERO chance this strong correlation happened by accident.'")
    print(f"   'We can be 99.99%+ confident that heating and cooling loads are truly related.'")

elif p_value < 0.001:
    print(f"   Very strong evidence against random chance")
elif p_value < 0.05:
    print(f"   Strong evidence against random chance")
else:
    print(f"   Weak evidence - could be random")

print(f"\n🎯 NULL HYPOTHESIS TESTING:")
print(f"   Null Hypothesis (H₀): 'Heating and cooling loads are NOT related (r = 0)'")
print(f"   Alternative Hypothesis (H₁): 'Heating and cooling loads ARE related (r ≠ 0)'")
print(f"   ")
print(f"   With p-value ≈ 0.0000:")
print(f"   🚫 REJECT H₀: We have overwhelming evidence to reject the null hypothesis")
print(f"   ✅ ACCEPT H₁: The relationship is statistically significant")

print(f"\n📈 REAL-WORLD IMPLICATIONS:")
print(f"   • This isn't just a number - it has practical meaning!")
print(f"   • We can confidently use one load type to predict the other")
print(f"   • Building designers can focus on factors affecting both loads simultaneously")
print(f"   • The relationship is so strong it's almost like a physical law for this dataset")

# Visual demonstration of what p-value means
print(f"\n🎨 VISUAL ANALOGY:")
print(f"   Imagine plotting 10,000 pairs of truly random, unrelated numbers:")
print(f"   📉📊📈📉📊 (most correlations would be near 0)")
print(f"   🎯 Our correlation of {corr:.4f} would be so extreme it would appear")
print(f"   🌟 less than once in those 10,000 random trials!")

# Sample size consideration
n_samples = len(energy_df)
print(f"\n📊 SAMPLE SIZE CONTEXT:")
print(f"   Sample size (n) = {n_samples}")
print(f"   Larger samples make p-values more reliable")
print(f"   With {n_samples} data points, our p-value calculation is very trustworthy")

# Practical vs Statistical significance
print(f"\n⚖️  STATISTICAL vs PRACTICAL SIGNIFICANCE:")
print(f"   📊 Statistical significance: p ≈ 0.0000 (✅ Very significant)")
print(f"   🏠 Practical significance: r = {corr:.4f} (✅ Very strong relationship)")
print(f"   🎯 Both agree: This is a meaningful, reliable relationship!")

# 4. Check for missing values (df.isnull().sum()), duplicates (df.duplicated().sum()), and data quality issues (e.g., negative values where impossible)

# Data Quality Checks
print("\n--- Data Quality Analysis ---")
print(f"Missing values:\n{energy_df.isnull().sum()}")
print(f"\nDuplicate rows: {energy_df.duplicated().sum()}")
print(f"\nNegative values check:")
for col in energy_df.select_dtypes(include=[np.number]).columns:
    neg_count = (energy_df[col] < 0).sum()
    if neg_count > 0:
        print(f"  {col}: {neg_count} negative values")


## Step 6: Energy Dataset - Heating vs Cooling Load Visualization


In [ ]:
"""
STEP 6: DETAILED LINE-BY-LINE EXPLANATION
=========================================

This step creates two visualizations to examine the relationship between
heating load and cooling load in the energy efficiency dataset.

PURPOSE: Visual analysis helps us understand:
1. How strongly these two variables are related
2. Whether the relationship is linear or non-linear
3. If there are any outliers or unusual patterns
4. The distribution and spread of the data points
"""

# LINE 1: Create a new figure with specified size
plt.figure(figsize=(15, 6))
"""
EXPLANATION:
- plt.figure(): Creates a new matplotlib figure object
- figsize=(15, 6): Sets the figure size to 15 inches wide by 6 inches tall
- This creates the canvas where our plots will be drawn
- Increased width to accommodate two subplots better
"""

# LINE 2: Create the first subplot (left side)
plt.subplot(1, 2, 1)
"""
EXPLANATION:
- plt.subplot(nrows, ncols, index): Divides the figure into a grid of subplots
- (1, 2, 1):
  * 1 row of subplots
  * 2 columns of subplots
  * 1 means this is the 1st subplot (left side)
- This creates a 1×2 grid layout: [Plot1][Plot2]
- We're now working on Plot1 (left side)
"""

# LINE 3: Create a scatter plot
sns.scatterplot(x='Heating_Load', y='Cooling_Load', data=energy_df, alpha=0.6, color='steelblue')
"""
EXPLANATION:
- sns.scatterplot(): Seaborn function to create a scatter plot
- x='Heating_Load': Column name for x-axis values
- y='Cooling_Load': Column name for y-axis values
- data=energy_df: The DataFrame containing our data
- alpha=0.6: Transparency level (0=invisible, 1=opaque)
  * 0.6 makes points semi-transparent so overlapping points are visible
  * Helps identify density patterns where many points cluster
- color='steelblue': Explicitly set point color to steelblue for consistency
"""

# LINE 4: Set title for first subplot
plt.title('Heating Load vs Cooling Load')

# LINE 5: Set x-axis label
plt.xlabel('Heating Load')

# LINE 6: Set y-axis label
plt.ylabel('Cooling Load')

# LINE 7: Create the second subplot (right side)
plt.subplot(1, 2, 2)

# LINE 8: Create a regression plot with explicit blue color
sns.regplot(x='Heating_Load', y='Cooling_Load', data=energy_df,
            scatter_kws={'alpha': 0.6, 'color': 'steelblue'},
            line_kws={'color': 'blue', 'linewidth': 2})
"""
EXPLANATION:
- sns.regplot(): Seaborn function that creates scatter plot + regression line
- x='Heating_Load': Same x-axis variable as before
- y='Cooling_Load': Same y-axis variable as before
- data=energy_df: Same DataFrame
- scatter_kws={'alpha': 0.6, 'color': 'steelblue'}:
  * Dictionary of keyword arguments for scatter points
  * alpha=0.6 makes the points semi-transparent (same as before)
  * color='steelblue' ensures consistent point color with left plot
- line_kws={'color': 'blue', 'linewidth': 2}:
  * Dictionary of keyword arguments for the regression line
  * color='blue' explicitly sets regression line to blue
  * linewidth=2 makes the line slightly thicker for better visibility

WHAT regplot DOES:
1. Creates scatter points (like scatterplot)
2. Fits a linear regression line through the points
3. Adds a confidence interval (shaded area) around the line
4. Shows both the data points AND the trend line in blue
"""

# LINE 9: Set title for second subplot
plt.title('Heating vs Cooling Load (with trend)')
"""
EXPLANATION:
- sets title for the second subplot
- "(with trend)" clarifies that this plot includes the regression line
- Distinguishes it from the first plot which only shows raw data points
"""

# LINE 10: Adjust layout and display
plt.tight_layout()
"""
EXPLANATION:
- plt.tight_layout(): Automatically adjusts subplot spacing
- Prevents overlapping titles, labels, or plots
- Optimizes the use of figure space
- Ensures both subplots fit nicely within the figure boundaries
- Very important when using multiple subplots
"""

# LINE 11: Display the complete figure
plt.show()

print("🎯 WHAT THESE VISUALIZATIONS REVEAL:")
print("=" * 50)
print("Left Plot (Scatter):")
print("  • Shows raw relationship between heating and cooling loads")
print("  • Each point represents one building")
print("  • Transparency helps see overlapping data points")
print("  • Pattern indicates correlation strength")
print("")
print("Right Plot (Regression):")
print("  • Same data with added trend analysis")
print("  • BLUE line = best-fit linear regression")
print("  • Light blue shaded area = confidence interval (uncertainty)")
print("  • Helps quantify the relationship strength")
print("")
print("Expected Observations:")
print(f"  • Strong positive correlation (r ≈ {corr:.3f})")
print("  • Points closely follow the blue trend line")
print("  • Narrow confidence interval indicates reliable relationship")
print("  • Few outliers far from the trend line")
print("")
print("🎨 COLOR SCHEME EXPLANATION:")
print("  • Scatter points: Steel blue (consistent across both plots)")
print("  • Regression line: Bright blue (stands out clearly)")
print("  • This color scheme ensures the trend line is clearly visible")


## Step 7: Energy Dataset - Statistical Summary


In [ ]:
"""
UNDERSTANDING STATISTICAL SUMMARY (describe() function)
======================================================

The describe() function provides essential statistical measures for numerical data.
Each row in the table represents a different statistical metric that helps us
understand the distribution, central tendency, and spread of our data.

Let's break down what each row means:
"""

# Statistical summary
print("--- Statistical Summary ---")
desc_stats = energy_df.describe()
print(desc_stats)

print("\n" + "=" * 80)
print("📊 DETAILED EXPLANATION OF EACH ROW IN THE STATISTICAL SUMMARY")
print("=" * 80)

print("🔢 ROW-BY-ROW EXPLANATION:")
print("─" * 50)

print("1. COUNT:")
print("   • MEANING: Total number of non-missing (valid) data points")
print("   • PURPOSE: Data completeness check")
print("   • INTERPRETATION:")
for col in ['Heating_Load', 'Cooling_Load']:
    count_val = desc_stats.loc['count', col]
    print(f"     - {col}: {count_val} valid observations")
print("   • If count < total rows, we have missing values to handle")

print(f"\n2. MEAN (Average):")
print("   • MEANING: Sum of all values ÷ number of values")
print("   • PURPOSE: Measure of central tendency (typical value)")
print("   • FORMULA: μ = Σx / n")
print("   • INTERPRETATION:")
for col in ['Heating_Load', 'Cooling_Load']:
    mean_val = desc_stats.loc['mean', col]
    print(f"     - {col}: {mean_val:.2f} (typical building uses this much energy)")
print("   • Higher mean suggests higher energy consumption overall")

print(f"\n3. STD (Standard Deviation):")
print("   • MEANING: Average distance of data points from the mean")
print("   • PURPOSE: Measure of variability/spread in the data")
print("   • FORMULA: σ = √(Σ(x - μ)² / n)")
print("   • INTERPRETATION:")
for col in ['Heating_Load', 'Cooling_Load']:
    std_val = desc_stats.loc['std', col]
    mean_val = desc_stats.loc['mean', col]
    cv = (std_val / mean_val) * 100  # Coefficient of variation
    print(f"     - {col}: {std_val:.2f}")
    print(f"       * About 68% of buildings fall within {mean_val - std_val:.1f} to {mean_val + std_val:.1f}")
    print(
        f"       * Coefficient of variation: {cv:.1f}% ({'high' if cv > 30 else 'moderate' if cv > 15 else 'low'} variability)")

print(f"\n4. MIN (Minimum Value):")
print("   • MEANING: Smallest value in the dataset")
print("   • PURPOSE: Lower boundary of the data range")
print("   • INTERPRETATION:")
for col in ['Heating_Load', 'Cooling_Load']:
    min_val = desc_stats.loc['min', col]
    print(f"     - {col}: {min_val:.2f} (most energy-efficient building)")
print("   • Useful for detecting impossible values (e.g., negative energy)")

print(f"\n5. 25% (First Quartile, Q1):")
print("   • MEANING: Value below which 25% of the data falls")
print("   • PURPOSE: Lower middle range boundary")
print("   • INTERPRETATION:")
for col in ['Heating_Load', 'Cooling_Load']:
    q1_val = desc_stats.loc['25%', col]
    print(f"     - {col}: {q1_val:.2f}")
    print(f"       * 25% of buildings use ≤ {q1_val:.1f} units (low energy consumers)")

print(f"\n6. 50% (Median, Q2):")
print("   • MEANING: Middle value when data is sorted (50th percentile)")
print("   • PURPOSE: Robust measure of central tendency")
print("   • INTERPRETATION:")
for col in ['Heating_Load', 'Cooling_Load']:
    median_val = desc_stats.loc['50%', col]
    mean_val = desc_stats.loc['mean', col]
    skew_direction = "right" if mean_val > median_val else "left" if mean_val < median_val else "symmetric"
    print(f"     - {col}: {median_val:.2f} (typical building)")
    print(f"       * Mean vs Median: {mean_val:.1f} vs {median_val:.1f} → {skew_direction} skewed")

print(f"\n7. 75% (Third Quartile, Q3):")
print("   • MEANING: Value below which 75% of the data falls")
print("   • PURPOSE: Upper middle range boundary")
print("   • INTERPRETATION:")
for col in ['Heating_Load', 'Cooling_Load']:
    q3_val = desc_stats.loc['75%', col]
    print(f"     - {col}: {q3_val:.2f}")
    print(f"       * 75% of buildings use ≤ {q3_val:.1f} units")
    print(f"       * 25% of buildings are high energy consumers (> {q3_val:.1f})")

print(f"\n8. MAX (Maximum Value):")
print("   • MEANING: Largest value in the dataset")
print("   • PURPOSE: Upper boundary of the data range")
print("   • INTERPRETATION:")
for col in ['Heating_Load', 'Cooling_Load']:
    max_val = desc_stats.loc['max', col]
    print(f"     - {col}: {max_val:.2f} (least energy-efficient building)")
print("   • Useful for detecting outliers or data entry errors")

print(f"\n🎯 KEY INSIGHTS FROM THE STATISTICS:")
print("=" * 50)

# Calculate additional insights
heating_range = desc_stats.loc['max', 'Heating_Load'] - desc_stats.loc['min', 'Heating_Load']
cooling_range = desc_stats.loc['max', 'Cooling_Load'] - desc_stats.loc['min', 'Cooling_Load']
iqr_heating = desc_stats.loc['75%', 'Heating_Load'] - desc_stats.loc['25%', 'Heating_Load']
iqr_cooling = desc_stats.loc['75%', 'Cooling_Load'] - desc_stats.loc['25%', 'Cooling_Load']

print(f"📏 DATA SPREAD:")
print(
    f"   • Heating Load range: {heating_range:.1f} units (from {desc_stats.loc['min', 'Heating_Load']:.1f} to {desc_stats.loc['max', 'Heating_Load']:.1f})")
print(
    f"   • Cooling Load range: {cooling_range:.1f} units (from {desc_stats.loc['min', 'Cooling_Load']:.1f} to {desc_stats.loc['max', 'Cooling_Load']:.1f})")
print(f"   • Middle 50% (IQR) - Heating: {iqr_heating:.1f}, Cooling: {iqr_cooling:.1f}")

print(f"\n🏠 PRACTICAL IMPLICATIONS:")
print(
    f"   • Average building needs more cooling ({desc_stats.loc['mean', 'Cooling_Load']:.1f}) than heating ({desc_stats.loc['mean', 'Heating_Load']:.1f})")
print(f"   • Energy efficiency varies significantly (7x range from min to max)")
print(f"   • Most buildings cluster in the middle range (median ≈ mean)")
print(f"   • Few extremely high or low energy consumers (relatively normal distribution)")

print(f"\n💡 WHAT THIS TELLS US ABOUT THE DATASET:")
print(f"   ✓ Complete data (no missing values)")
print(f"   ✓ Reasonable value ranges (no negative or impossible values)")
print(f"   ✓ Good variability for machine learning (not all identical)")
print(f"   ✓ Balanced distribution (mean ≈ median suggests no extreme skewness)")


## Step 8: Energy Dataset - Feature Distribution Analysis


In [ ]:
"""
STEP 8: FEATURE DISTRIBUTION ANALYSIS - SIMPLE EXPLANATION
===========================================================

PURPOSE: Create histograms to see how each feature's values are spread out
- Are values normally distributed (bell curve)?
- Are there any unusual patterns or gaps?
- Do we have outliers or extreme values?
"""

# Define which features we want to analyze (exclude target variables)
numerical_features = ['Relative_Compactness', 'Surface_Area', 'Wall_Area',
                      'Roof_Area', 'Overall_Height', 'Glazing_Area']

# Create a grid of subplots (2 rows, 3 columns = 6 plots total)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
"""
EXPLANATION:
- plt.subplots(2, 3): Creates 2 rows and 3 columns of plots
- figsize=(15, 10): Makes the overall figure 15 inches wide, 10 inches tall
- This gives us space for 6 individual histogram plots
"""

# Convert 2D array of axes to 1D for easier indexing
axes = axes.ravel()
"""
EXPLANATION:
- subplots() creates a 2D array: [[ax1, ax2, ax3], [ax4, ax5, ax6]]
- ravel() flattens it to 1D: [ax1, ax2, ax3, ax4, ax5, ax6]
- This makes it easier to loop through with a simple index
"""

# Create a histogram for each feature
for i, feat in enumerate(numerical_features):
    """
    EXPLANATION:
    - enumerate() gives us both index (i) and feature name (feat)
    - i = 0,1,2,3,4,5 for each feature
    - feat = actual feature names like 'Relative_Compactness', etc.
    """

    # Create histogram with density curve overlay
    sns.histplot(energy_df[feat], kde=True, ax=axes[i])
    """
    EXPLANATION:
    - sns.histplot(): Creates a histogram (bar chart showing frequency)
    - energy_df[feat]: Gets the data for this specific feature
    - kde=True: Adds a smooth density curve (Kernel Density Estimation)
    - ax=axes[i]: Puts this plot in the i-th subplot position

    WHAT WE SEE:
    - Bars: How many buildings have each value range
    - Curve: Smooth estimate of the distribution shape
    """

    # Set title for this subplot
    plt.title(f'Distribution of {feat}')
    """
    EXPLANATION:
    - set_title(): Adds a title above each plot
    - f'Distribution of {feat}': Uses f-string to insert feature name
    - Example: "Distribution of Relative_Compactness"
    """

    # Rotate x-axis labels to prevent overlap
    axes[i].tick_params(axis='x', rotation=45)
    """
    EXPLANATION:
    - tick_params(): Adjusts the appearance of axis labels
    - axis='x': Only affect x-axis labels
    - rotation=45: Rotate labels 45 degrees to prevent overlapping
    - This makes long feature names more readable
    """

# Automatically adjust spacing between plots
plt.tight_layout()
"""
EXPLANATION:
- Prevents plots from overlapping each other
- Adjusts margins and spacing automatically
- Makes the overall figure look cleaner and more professional
"""

# Display all the plots
plt.show()
"""
EXPLANATION:
- Actually shows the complete figure with all 6 histograms
- Without this, the plots would be created but not displayed
"""

print("🔍 WHAT TO LOOK FOR IN THESE DISTRIBUTIONS:")
print("=" * 50)
print("✅ GOOD SIGNS:")
print("   • Bell-shaped curves (normal distribution)")
print("   • Smooth, continuous distributions")
print("   • Reasonable value ranges")
print("   • No large gaps in the data")
print("")
print("⚠️  POTENTIAL ISSUES:")
print("   • Highly skewed distributions (long tails)")
print("   • Multiple peaks (bimodal)")
print("   • Extreme outliers")
print("   • Gaps or missing value ranges")
print("")
print("💡 PRACTICAL MEANING:")
print("   • Each histogram shows how building characteristics vary")
print("   • Normal distributions are easier for models to learn")
print("   • Unusual patterns might need special handling")


## Step 9: Energy Dataset - Correlation Matrix


In [ ]:
# Feature correlation analysis
# Analyze relationships between numerical features (-1 to +1 scale)

plt.figure(figsize=(12, 10))  # Large canvas for readability

# Calculate Pearson correlation matrix (measures linear relationships)
# Alternative: .corr(method='spearman') for non-linear monotonic relationships
correlation_matrix = energy_df.corr()

# Create color-coded heatmap
sns.heatmap(
    correlation_matrix,
    annot=True,  # Display correlation values in cells
    cmap='coolwarm',  # Blue=negative, red=positive correlation
    center=0,  # White=no correlation
    square=True,
    linewidths=0.5
)

plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

# Interpretation guide:
# +1.0: Perfect positive correlation | -1.0: Perfect negative correlation
# 0.0: No linear relationship | >0.7 or <-0.7: Strong correlation (potential multicollinearity)


## Step 10: Energy Dataset - Categorical Features Analysis


In [ ]:
"""
STEP 10: CATEGORICAL FEATURES ANALYSIS
======================================
"""
# ─────────────────────────────────────────────────────────────────────────────
# Goal: Inspect categorical columns and convert them into numeric representations
#       suitable for ML models via one-hot encoding.
# Why: Most scikit-learn estimators accept only numeric arrays; categories must
#      be expanded into indicator (0/1) columns to avoid ordinal assumptions.
# ─────────────────────────────────────────────────────────────────────────────

print("--- Categorical Feature Analysis ---")  # User-facing marker for this step

# Show distinct categories present in 'Orientation' to sanity-check the raw labels
# • Helps confirm the domain (e.g., {2,3,4,5}) and detect unexpected values
print(f"Unique values in Orientation: {energy_df['Orientation'].unique()}")

# Show distinct categories present in 'Glazing_Area_Distribution' for the same reasons
# • Verifies the domain and informs the number of dummy columns that will be created
print(f"Unique values in Glazing_Area_Distribution: {energy_df['Glazing_Area_Distribution'].unique()}")

# Apply one-hot encoding to the two categorical columns:
# • 'columns=[...]' specifies which columns to expand into dummies
# • 'prefix=[...]' controls the column name prefixes to keep features interpretable
# • Output: original columns replaced by multiple binary columns per category value
energy_encoded = pd.get_dummies(
    energy_df,  # Source DataFrame
    columns=['Orientation', 'Glazing_Area_Distribution'],  # Categorical cols
    prefix=['Orient', 'Glaz_Dist']  # Name prefixes for new dummies
)

# Print the post-encoding shape so we see how many columns were added
# (useful to estimate model dimensionality and detect feature explosions)
print(f"Shape after encoding: {energy_encoded.shape}")


## Step 11: Energy Dataset - Outlier Detection


In [ ]:
"""
STEP 11: OUTLIER DETECTION
==========================
"""
# Outlier visualization with boxplots

# Create a grid of 2 rows × 4 columns of subplots.
# plt.subplots returns (figure, axes) where:
#   2, 4               → grid layout (rows, columns)
#   figsize=(16, 8)    → width=16 inches, height=8 inches; increases readability
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# axes is a 2D array [ [ax00, ax01, ...], [ax10, ax11, ...] ].
# .ravel() flattens to 1D (length 8) so we can index with i in a loop.
axes = axes.ravel()

# Features to inspect for outliers—includes both predictors and targets
features_to_check = [
    'Relative_Compactness', 'Surface_Area', 'Wall_Area', 'Roof_Area',
    'Overall_Height', 'Glazing_Area', 'Heating_Load', 'Cooling_Load'
]

for i, feat in enumerate(features_to_check):
    # seaborn.boxplot draws a box-and-whisker chart.
    # params:
    #   y=energy_df[feat] → the data series to plot vertically (box aligns with y-axis)
    #   ax=axes[i]        → explicit Matplotlib Axes object to draw on (the i-th subplot)
    # behavior:
    #   • box shows IQR [Q1,Q3]
    #   • line inside box is median
    #   • whiskers extend to data within 1.5×IQR
    #   • points beyond whiskers are plotted as outliers
    sns.boxplot(y=energy_df[feat], ax=axes[i])

    # Title for this subplot; helps identify which feature is displayed.
    axes[i].set_title(f'Boxplot: {feat}')

# Adjust spacing between subplots to avoid overlapping titles/labels.
# tight_layout() auto-computes paddings based on content.
plt.tight_layout()

plt.show()


## Step 12: Energy Dataset - Feature Engineering


In [ ]:
"""
STEP 12: FEATURE ENGINEERING
============================
"""
# Feature engineering: add domain-inspired signals

print("--- Feature Engineering ---")

# Sum of three area-related columns → a single "total envelope area" indicator.
# Uses vectorized column addition; aligns by index automatically.
energy_encoded['Total_Area'] = (
    energy_encoded['Surface_Area'] +
    energy_encoded['Wall_Area'] +
    energy_encoded['Roof_Area']
)

# A rough proxy for building "volume": surface area × overall height.
# (Not geometric volume but often positively correlated with energy needs.)
energy_encoded['Volume_Proxy'] = (
    energy_encoded['Surface_Area'] *
    energy_encoded['Overall_Height']
)

# Shape–size interaction: relative compactness divided by height.
# Assumes Overall_Height > 0 (true in this dataset).
energy_encoded['Compactness_Height_Ratio'] = (
    energy_encoded['Relative_Compactness'] /
    energy_encoded['Overall_Height']
)

print("✓ Engineered features created:")
print("- Total_Area = Surface_Area + Wall_Area + Roof_Area")
print("- Volume_Proxy = Surface_Area * Overall_Height")
print("- Compactness_Height_Ratio = Relative_Compactness / Overall_Height")


## Step 13: Energy Dataset - Prepare for Modeling


In [ ]:
"""
STEP 13: PREPARE FOR MODELING
=============================
"""
# Train/test preparation (features/target split + holdout)

# Build list of predictor column names by excluding both targets.
# List comprehension iterates over columns and filters out 'Heating_Load' and 'Cooling_Load'.
features_for_model = [
    col for col in energy_encoded.columns
    if col not in ['Heating_Load', 'Cooling_Load']
]

# X: 2D DataFrame of features (n_samples × n_features)
X = energy_encoded[features_for_model]

# y: 1D Series of target values (Heating_Load)
y = energy_encoded['Heating_Load']

print(f"Features for modeling: {len(features_for_model)}")
print(f"Target variable: Heating_Load")

# Split into training and test sets.
# params:
#   test_size=0.2          → 20% of the data becomes the test split
#   random_state=STUDENT_SEED
#                         → makes the split reproducible (same student seed → same split)
#                         → Controls the random number generator used to shuffle the data before splitting
#                         → Same random_state value = identical train/test splits every time you run the code
#                         → Different random_state values = different random splits
#   stratify=None          → not applicable because y is continuous (no class stratification)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=STUDENT_SEED, stratify=None
)

# What X_train.shape returns:
# First number: Number of training samples (rows)
# Second number: Number of features (columns)
print(f"Training set shape: {X_train.shape}")  # (n_train, n_features)
print(f"Test set shape: {X_test.shape}")        # (n_test, n_features)


## Step 14: Energy Dataset - Feature Scaling


In [ ]:
"""
STEP 14: FEATURE SCALING
========================
Feature scaling is essential for:
• Gradient-based optimization algorithms (e.g., SGD) to converge faster
• Regularization methods (e.g., Ridge, Lasso) to work effectively
• Ensuring all features contribute equally to distance computations (e.g., in KNN, SVM)
• Improving model interpretability by placing features on a common scale

SCALING STRATEGY:
We use StandardScaler which:
• Centers features by removing the mean
• Scales features to unit variance (dividing by standard deviation)
• Transforms the data to have a standard normal distribution (mean=0, std=1)
"""
print("--- Feature Scaling ---")

# Create a StandardScaler instance.
# It will learn μ (mean) and σ (std) for each feature from the training data only.
scaler = StandardScaler()

# Fit on training features and transform them.
# .fit_transform:
#   • computes μ and σ per column from X_train
#   • applies (x - μ) / σ
X_train_scaled = scaler.fit_transform(X_train)

# Transform test features using the SAME μ and σ learned from training.
# .transform (no .fit!) prevents information leakage.
X_test_scaled = scaler.transform(X_test)

print("✓ Features scaled using StandardScaler")
# Quick sanity checks: training-scaled features should have mean≈0, std≈1 per column.
print(f"Training set mean (first 5): {X_train_scaled.mean(axis=0)[:5].round(4)}")
print(f"Training set std (first 5): {X_train_scaled.std(axis=0)[:5].round(4)}")


## Step 15: Energy Dataset - Model Evaluation Function


In [ ]:
"""
STEP 15: MODEL EVALUATION FUNCTION
==================================
"""
# Generic evaluation helper for regression models

def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """Train model, generate predictions, and compute regression metrics."""
    # Fit the model on training data (estimating parameters).
    model.fit(X_train, y_train)

    # Predict target values for the test set (held-out data).
    y_pred = model.predict(X_test)

    # Error metrics:
    # mean_squared_error(y_true, y_pred) → average of (error)^2
    mse = mean_squared_error(y_test, y_pred)

    # Root of MSE to bring error back to target units (same units as y).
    rmse = np.sqrt(mse)

    # mean_absolute_error(y_true, y_pred) → average absolute deviation
    mae = mean_absolute_error(y_test, y_pred)

    # r2_score(y_true, y_pred) → proportion of variance explained (can be negative)
    r2 = r2_score(y_test, y_pred)

    return {
        'model': model,          # return the fitted estimator for later use (e.g., coefficients, predict)
        'predictions': y_pred,   # cached predictions to avoid recomputing
        'mse': mse,
        'rmse': rmse,
        'mae': mae,
        'r2': r2
    }

print("✓ Model evaluation function defined")


## Step 16: Energy Dataset - Train Linear Regression


In [ ]:
"""
STEP 16: TRAIN LINEAR REGRESSION
================================
"""
# Baseline: Linear Regression (OLS)

print("--- Model Training and Evaluation ---")
energy_results = {}  # dict to store metrics by model name

print("Training Linear Regression...")
lr_model = LinearRegression()  # Ordinary Least Squares; no regularization

# Use the scaled features for numerically stable coefficients/solver steps.
energy_results['Linear'] = evaluate_model(
    lr_model,
    X_train_scaled,   # scaled training X
    X_test_scaled,    # scaled test X
    y_train,          # unscaled y (target usually not scaled for interpretability)
    y_test,
    'Linear Regression'
)

# Report two key metrics: R² (higher is better) and RMSE (lower is better).
print(f"✓ Linear Regression - R²: {energy_results['Linear']['r2']:.4f}, "
      f"RMSE: {energy_results['Linear']['rmse']:.4f}")


## Step 17: Energy Dataset - Train Polynomial Regression


In [ ]:
"""
STEP 17: TRAIN POLYNOMIAL REGRESSION
====================================
"""
# Polynomial Regression via Pipeline (degrees 2–4)

polynomial_degrees = [2, 3, 4]

for degree in polynomial_degrees:
    print(f"Training Polynomial Regression (degree {degree})...")

    # sklearn.pipeline.Pipeline chains multiple steps into one estimator.
    # Steps:
    #   ('poly', PolynomialFeatures(...)) → expands columns with polynomial terms
    #   ('scaler', StandardScaler())      → standardizes the expanded feature space
    #   ('reg', LinearRegression())       → OLS in the transformed space
    #
    # PolynomialFeatures params:
    #   degree=degree       → maximum polynomial degree
    #   include_bias=False  → do NOT add a constant 1 column (the regressor adds intercept)
    poly_pipeline = Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
        ('scaler', StandardScaler()),
        ('reg', LinearRegression())
    ])

    # Evaluate with raw X (not pre-scaled) because the pipeline contains its own scaler.
    energy_results[f'Poly_{degree}'] = evaluate_model(
        poly_pipeline,
        X_train, X_test, y_train, y_test,
        f'Polynomial {degree}'
    )

    print(f"✓ Polynomial {degree} - R²: {energy_results[f'Poly_{degree}']['r2']:.4f}, "
          f"RMSE: {energy_results[f'Poly_{degree}']['rmse']:.4f}")


## Step 18: Energy Dataset - Results Summary


In [ ]:
"""
STEP 18: RESULTS SUMMARY
========================
"""
# Summarize model performances and pick best by R²

print("--- Energy Efficiency Results ---")
# Nice fixed-width header: column labels with left alignment and width specs
print(f"{'Model':<20} {'R²':<8} {'RMSE':<8} {'MAE':<8} {'MSE':<10}")
print("-" * 60)

# Iterate dict items; format floats to 4 decimals for readability
for model_name, results in energy_results.items():
    print(f"{model_name:<20} {results['r2']:<8.4f} {results['rmse']:<8.4f} "
          f"{results['mae']:<8.4f} {results['mse']:<10.4f}")

# max(..., key=lambda ...) selects the key (model name) with the largest R²
energy_best = max(energy_results.keys(), key=lambda x: energy_results[x]['r2'])
print(f"\n🏆 Best Energy Model: {energy_best} with R² = {energy_results[energy_best]['r2']:.4f}")


## Step 19: Energy Dataset - Feature Importance Analysis


In [ ]:
"""
STEP 19: FEATURE IMPORTANCE ANALYSIS
====================================
"""
# Linear-model feature importance (coefficients on scaled features)

print("--- Feature Importance Analysis ---")

# Build a DataFrame mapping each feature to its learned coefficient.
# energy_results['Linear']['model'] is the fitted LinearRegression on X_train_scaled.
feature_importance = pd.DataFrame({
    'feature': features_for_model,
    'coefficient': energy_results['Linear']['model'].coef_
})

# Absolute value allows ranking by magnitude regardless of positive/negative sign.
feature_importance['abs_coefficient'] = np.abs(feature_importance['coefficient'])

# Sort descending by |coef| so most influential features appear first.
feature_importance = feature_importance.sort_values('abs_coefficient', ascending=False)

print(f"Top 5 most important features:")
for i, row in feature_importance.head().iterrows():
    print(f"  {row['feature']}: {row['coefficient']:.4f}")


## Step 20: Energy Dataset - Visualization


In [ ]:
"""
STEP 20: VISUALIZATION
=====================
"""
# Visualizations: feature importance (bar) and predicted vs actual (scatter)

plt.figure(figsize=(12, 8))  # overall canvas for both subplots

# Subplot 1: top-10 absolute coefficients
plt.subplot(1, 2, 1)
top_features = feature_importance.head(10)

# seaborn.barplot draws a bar chart.
# params (common):
#   data=top_features          → DataFrame providing columns
#   x='abs_coefficient'        → numeric column on X-axis (bar length)
#   y='feature'                → categorical column on Y-axis (bar labels)
sns.barplot(data=top_features, x='abs_coefficient', y='feature')

plt.title('Top 10 Feature Importance (Linear Regression)')  # readable title
plt.xlabel('Absolute Coefficient Value')                    # axis label (units: scaled)

# Subplot 2: predicted vs actual for the chosen best model
plt.subplot(1, 2, 2)

# Retrieve cached predictions from the best model we picked earlier.
best_predictions = energy_results[energy_best]['predictions']

# Matplotlib scatter:
#   x=y_test                   → true target values on x-axis
#   y=best_predictions         → predicted target values on y-axis
#   alpha=0.6                  → semi-transparency so dense areas show density
plt.scatter(y_test, best_predictions, alpha=0.6)

# Add a 45° reference line for "perfect predictions".
#   [y_test.min(), y_test.max()] → x-coordinates spanning the data range
#   [y_test.min(), y_test.max()] → y-coordinates equal to x (identity line)
#   'r--'                        → red dashed line style
#   lw=2                         → line width 2 points
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)

plt.xlabel('Actual Heating Load')
plt.ylabel('Predicted Heating Load')
plt.title(f'Predicted vs Actual ({energy_best})')

plt.tight_layout()  # adjust spacing to avoid overlap
plt.show()


# PART 2: WINE QUALITY ASSESSMENT ANALYSIS


## Step 29: Wine Dataset - Load Data


In [ ]:
"""
STEP 29: LOAD WINE DATA
=======================
"""
print("=" * 60)
print("PART 2: WINE QUALITY ASSESSMENT ANALYSIS")
print("=" * 60)

# Load Wine Quality Dataset from UCI repository
print("Loading Wine Quality Dataset...")
try:
    wine_df = pd.read_csv(
        'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv',
        sep=';'
    )
    print(f"✓ Dataset loaded successfully! Shape: {wine_df.shape}")  # (rows, columns)
except Exception as e:
    print(f"✗ Error loading dataset: {e}")

# .info() prints dtypes, non-null counts; useful for null/memory checks.
print("\nDataset Info:")
print(wine_df.info())

# Quick peek at first 5 rows to confirm parsing and column names.
print("\nFirst 5 rows:")
print(wine_df.head())


## Step 30: Wine Dataset - Data Quality Analysis


In [ ]:
"""
STEP 30: WINE DATA QUALITY ANALYSIS
===================================
"""
# Basic data quality diagnostics for the wine dataset.

print("--- Data Quality Analysis ---")

# .isnull().sum() counts nulls per column; should be all zeros in this dataset.
print(f"Missing values:\n{wine_df.isnull().sum()}")

# .duplicated().sum() counts rows with identical values across all columns.
print(f"\nDuplicate rows: {wine_df.duplicated().sum()}")

# Target variable ('quality') distribution to understand class balance (for regression it's ordinal/continuous-like).
print(f"\n--- Target Variable Analysis ---")
print(f"Quality score distribution:\n{wine_df['quality'].value_counts().sort_index()}")
print(f"Quality range: {wine_df['quality'].min()} to {wine_df['quality'].max()}")
print(f"Quality mean: {wine_df['quality'].mean():.2f} ± {wine_df['quality'].std():.2f}")


## Step 31: Wine Dataset - Quality Distribution Visualization


In [ ]:
"""
STEP 31: WINE QUALITY DISTRIBUTION VISUALIZATION
================================================
"""
# Visualize target ('quality') distribution in two ways for complementary insight.

plt.figure(figsize=(12, 6))         # 12x6 inches overall canvas

# Left subplot: count of each discrete 'quality' label.
plt.subplot(1, 2, 1)
# seaborn.countplot parameters:
#   data=wine_df       → source DataFrame
#   x='quality'        → column whose distinct values to count on the x-axis
sns.countplot(data=wine_df, x='quality')
plt.title('Wine Quality Distribution')
plt.xlabel('Quality Score')
plt.ylabel('Count')

# Right subplot: histogram approximates distribution as continuous; KDE overlays a smooth density.
plt.subplot(1, 2, 2)
# seaborn.histplot parameters:
#   data series → wine_df['quality'] (1D array-like)
#   kde=True    → add kernel density estimate curve
#   bins=6      → number of histogram bins; here aligns roughly to range of integer scores
sns.histplot(wine_df['quality'], kde=True, bins=6)
plt.title('Wine Quality Distribution (Histogram)')
plt.xlabel('Quality Score')
plt.ylabel('Frequency')

plt.tight_layout()  # avoid overlaps between subplots
plt.show()


## Step 32: Wine Dataset - Statistical Summary


In [ ]:
"""
STEP 32: WINE STATISTICAL SUMMARY
=================================
"""
# Print descriptive statistics across all numeric columns to check scale, spread, and potential outliers.

print("--- Statistical Summary ---")
print(wine_df.describe())


## Step 33: Wine Dataset - Feature Distribution Analysis


In [ ]:
"""
STEP 33: WINE FEATURE DISTRIBUTION ANALYSIS
===========================================
"""
# Distribution plots for all input features (excluding target 'quality') in a grid layout.

# Build a list of feature columns (predictors only).
wine_features = [col for col in wine_df.columns if col != 'quality']

n_features = len(wine_features)      # total number of predictors
n_cols = 4                           # number of subplots per row
# ceil division to compute needed rows: (n_features + n_cols - 1) // n_cols
n_rows = (n_features + n_cols - 1) // n_cols

# plt.subplots returns (figure, axes) with specified grid; figsize scales with rows
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.ravel()  # Flatten 2D array to 1D for easy indexing

for i, feat in enumerate(wine_features):
    if i < len(axes):
        # seaborn.histplot parameters:
        #   kde=True → overlay kernel density estimate
        #   ax=axes[i] → draw on i-th subplot
        sns.histplot(wine_df[feat], kde=True, ax=axes[i])
        axes[i].set_title(f'Distribution of {feat}')
        axes[i].tick_params(axis='x', rotation=45)  # rotate tick labels to reduce overlap

# If grid has extra empty axes (when features % n_cols != 0), hide them for a clean look.
for i in range(len(wine_features), len(axes)):
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## Step 34: Wine Correlation Analysis


In [ ]:
"""
STEP 34: WINE CORRELATION ANALYSIS
==================================
"""
# Feature correlation matrix to identify linear associations and potential multicollinearity.

plt.figure(figsize=(12, 10))

# .corr() computes Pearson correlation by default for numeric columns only.
wine_corr = wine_df.corr()

# seaborn.heatmap parameters:
#   data=wine_corr    → correlation matrix (square DataFrame)
#   annot=True        → write correlation coefficients in each cell
#   cmap='RdBu_r'     → diverging colormap; red/blue with reversed orientation
#   center=0          → white midpoint at zero correlation
#   square=True       → force square cells for aesthetics
#   linewidths=0.5    → thin lines between cells for readability
sns.heatmap(wine_corr, annot=True, cmap='RdBu_r', center=0, square=True, linewidths=0.5)

plt.title('Wine Features Correlation Matrix')
plt.tight_layout()
plt.show()

## Step 35: Wine Quality Correlation Analysis


In [ ]:
"""
STEP 35: WINE QUALITY CORRELATION ANALYSIS
==========================================
"""
# Print correlation of each feature with the target 'quality' to see rough linear influence.

print("--- Quality Correlation Analysis ---")
# wine_df.corr()['quality'] → selects the 'quality' column of the correlation matrix
quality_corr = wine_df.corr()['quality'].sort_values(ascending=False)
print("Features most correlated with quality:")
for feature, corr_val in quality_corr.items():
    if feature != 'quality':
        print(f"  {feature}: {corr_val:.4f}")

## Step 36: Wine Dataset - Outlier Detection


In [ ]:
"""
STEP 36: WINE OUTLIER DETECTION
===============================
"""
# Outlier visualization with boxplots

print("--- Outlier Analysis ---")

fig, axes = plt.subplots(3, 4, figsize=(16, 12))  # 3 rows × 4 cols grid
axes = axes.ravel()

for i, feat in enumerate(wine_features):
    if i < len(axes):
        # seaborn.boxplot parameters:
        #   y=wine_df[feat] → vertical box for feature values
        #   ax=axes[i]      → draw on i-th subplot
        # Behavior (boxplot anatomy) identical to earlier explanation in Energy section.
        sns.boxplot(y=wine_df[feat], ax=axes[i])
        axes[i].set_title(f'Boxplot: {feat}')

plt.tight_layout()
plt.show()


## Step 37: Wine Dataset - Remove Outliers


In [ ]:
"""
STEP 37: WINE REMOVE OUTLIERS
=============================
"""
# Outlier removal helper using the IQR (Interquartile Range) rule.

def remove_outliers_iqr(df, columns):
    """Remove outliers using the 1.5 * IQR rule for specified columns."""
    df_clean = df.copy()     # work on a copy to avoid mutating original frame
    outliers_removed = 0     # counter for removed rows (across all specified columns)

    for column in columns:
        # .quantile(0.25/0.75) gives Q1 and Q3 for the column
        Q1 = df_clean[column].quantile(0.25)
        Q3 = df_clean[column].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR   # typical Tukey rule lower fence
        upper_bound = Q3 + 1.5 * IQR   # typical Tukey rule upper fence

        before_count = len(df_clean)
        # Keep only rows within [lower_bound, upper_bound] for this column.
        # Note: filtering is cumulative across columns, so rows outlying in any selected
        # column will be dropped.
        df_clean = df_clean[(df_clean[column] >= lower_bound) & (df_clean[column] <= upper_bound)]
        after_count = len(df_clean)
        outliers_removed += before_count - after_count

    return df_clean, outliers_removed

# Choose a subset of columns known to have long tails; reduces extreme leverage in regression.
outlier_features = ['residual sugar', 'free sulfur dioxide', 'total sulfur dioxide']

# Apply function and report the number of removed rows plus new shape.
wine_clean, outliers_count = remove_outliers_iqr(wine_df, outlier_features)
print(f"✓ Outliers removed: {outliers_count}")
print(f"Dataset shape after outlier removal: {wine_clean.shape}")

## Step 38: Wine Dataset - Feature Engineering


In [ ]:
"""
STEP 38: WINE FEATURE ENGINEERING
=================================
Feature engineering adds new predictive signals based on domain knowledge.
For wine quality prediction, we create:
1. Ratios: Indicate balance between components (e.g., free vs. total sulfur dioxide)
2. Composite indicators: Summarize multiple related features (e.g., total acidity)
3. Interaction terms: Capture combined effects of features

WHY FEATURE ENGINEERING?
• Improve model performance by providing more informative features
• Help models capture underlying data patterns and relationships
• Reduce the risk of underfitting by expanding the feature space
"""
print("--- Feature Engineering ---")
wine_engineered = wine_clean.copy()  # non-destructive copy for adding columns

# acid_ratio: fixed / volatile acidity
#  - Higher fixed acidity with lower volatile acidity could indicate different sensory profiles.
wine_engineered['acid_ratio'] = wine_engineered['fixed acidity'] / wine_engineered['volatile acidity']

# sulfur_ratio: free / total sulfur dioxide
#  - Captures proportion of active preservative (free SO2) relative to total; could relate to quality/stability.
wine_engineered['sulfur_ratio'] = wine_engineered['free sulfur dioxide'] / wine_engineered['total sulfur dioxide']

# total_acidity: simple sum of key acidity components
#  - Composite indicator of acidity that might relate to perceived taste and quality.
wine_engineered['total_acidity'] = (
    wine_engineered['fixed acidity'] +
    wine_engineered['volatile acidity'] +
    wine_engineered['citric acid']
)

print("✓ Engineered features created:")
print("- acid_ratio = fixed acidity / volatile acidity")
print("- sulfur_ratio = free sulfur dioxide / total sulfur dioxide")
print("- total_acidity = fixed + volatile + citric acid")

## Step 39: Wine Dataset - Prepare for Modeling


In [ ]:
"""
STEP 39: WINE PREPARE FOR MODELING
==================================
With feature engineering complete, we prepare the wine dataset for modeling:
1. Handle infinite values: Replace ±inf with NaN, then impute NaNs
2. Split data: Separate features (X) and target (y) using selected features
3. Train-test split: Split data into training and testing sets
4. Feature scaling: Standardize features to have mean=0, std=1
"""
# Build training data (X) and target (y) for wine; handle infs from division, then split.

# All columns except 'quality' are predictors.
wine_features_for_model = [col for col in wine_engineered.columns if col != 'quality']

X_wine = wine_engineered[wine_features_for_model]  # feature matrix
y_wine = wine_engineered['quality']                # target: quality score

# Division may produce ±inf (e.g., division by zero) and NaNs.
# Replace infinities with NaN, then impute NaNs with column medians.
X_wine = X_wine.replace([np.inf, -np.inf], np.nan)
X_wine = X_wine.fillna(X_wine.median())

print(f"Features for modeling: {len(wine_features_for_model)}")
print(f"Target variable: quality")

# train_test_split parameters:
#   test_size=0.2            → 20% holdout
#   random_state=STUDENT_SEED→ reproducibility
#   stratify=y_wine          → ensures target distribution is similar in train/test (useful here as quality is discrete)
X_wine_train, X_wine_test, y_wine_train, y_wine_test = train_test_split(
    X_wine, y_wine, test_size=0.2, random_state=STUDENT_SEED, stratify=y_wine
)

print(f"Wine training set shape: {X_wine_train.shape}")
print(f"Wine test set shape: {X_wine_test.shape}")

# Feature scaling
"""
WHY FEATURE SCALING?
• Standardization (Z-score scaling) centers features to have mean=0 and std=1
• Important for algorithms that rely on distance calculations (e.g., KNN, SVM)
• Helps gradient-based optimizers converge faster
• Regularization methods (e.g., Ridge, Lasso) perform better with standardized data
"""
# Create a new StandardScaler instance.
music_scaler = StandardScaler()

# Fit scaler on training data and transform it in one step.
X_wine_train_scaled = music_scaler.fit_transform(X_wine_train)

# Transform the test set using μ and σ from training to avoid leakage
X_wine_test_scaled = music_scaler.transform(X_wine_test)

print("✓ Wine features scaled using StandardScaler")

# Verify scaling worked correctly
print(f"Training set mean (first 5 features): {X_wine_train_scaled.mean(axis=0)[:5].round(4)}")
print(f"Training set std (first 5 features): {X_wine_train_scaled.std(axis=0)[:5].round(4)}")

## Step 41: Wine Dataset - Train Linear Regression


In [ ]:
"""
STEP 41: WINE TRAIN LINEAR REGRESSION
=====================================
"""
# Model Training for Wine Dataset

print("--- Wine Model Training and Evaluation ---")

# Container (dict) to store model objects and their evaluation metrics keyed by a label.
wine_results = {}

print("Training Linear Regression for Wine...")
# LinearRegression() uses ordinary least squares (no regularization).
wine_lr = LinearRegression()

# Evaluate on the scaled design matrices for numeric stability.
wine_results['Linear'] = evaluate_model(
    wine_lr,
    X_wine_train_scaled,   # features: scaled train
    X_wine_test_scaled,    # features: scaled test
    y_wine_train,          # targets: train (kept unscaled for interpretability)
    y_wine_test,           # targets: test
    'Linear Regression'    # label for tracking/reporting
)

# Display results
print(f"✓ Wine Linear Regression - R²: {wine_results['Linear']['r2']:.4f}, RMSE: {wine_results['Linear']['rmse']:.4f}")

# Additional insight about the baseline performance
baseline_r2 = wine_results['Linear']['r2']
baseline_rmse = wine_results['Linear']['rmse']

print(f"\n📊 BASELINE PERFORMANCE ANALYSIS:")
print(f"   • R² = {baseline_r2:.4f} means model explains {baseline_r2*100:.2f}% of year variance")
print(f"   • RMSE = {baseline_rmse:.2f} years average prediction error")
print(f"   • For a dataset spanning ~{y_wine_test.max() - y_wine_test.min():.0f} years, this represents {'good' if baseline_rmse < 10 else 'moderate' if baseline_rmse < 20 else 'poor'} accuracy")

## Step 42: Wine Dataset - Train Polynomial Regression


In [ ]:
"""
STEP 42: WINE TRAIN POLYNOMIAL REGRESSION
=========================================
"""
# Polynomial Regression for Wine using a Pipeline:
# Pipeline structure: [('poly', PolynomialFeatures), ('scaler', StandardScaler), ('reg', LinearRegression)]
for degree in [2, 3, 4]:
    print(f"Training Polynomial Regression (degree {degree}) for Wine...")

    # PolynomialFeatures parameters:
    #   degree=degree      : include all polynomial terms up to the specified degree (incl. interactions/powers)
    #   include_bias=False : do NOT add constant 1 column (LinearRegression has its own intercept)
    # StandardScaler()     : standardize expanded feature space
    # LinearRegression()   : OLS on transformed features
    wine_poly_pipeline = Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
        ('scaler', StandardScaler()),
        ('reg', LinearRegression())
    ])

    # Evaluate with a subset for computational efficiency
    if degree > 2:
        # Use subset of available samples for degrees > 2 to manage computational complexity
        # Ensure we don't try to sample more than available
        max_train_subset = min(10000, len(X_wine_train))
        max_test_subset = min(1000, len(X_wine_test))

        print(f"   Using subset: {max_train_subset} train samples, {max_test_subset} test samples")

        # Only subset if we have more data than needed
        if len(X_wine_train) > max_train_subset:
            train_subset_indices = np.random.choice(len(X_wine_train), max_train_subset, replace=False)
            X_subset = X_wine_train.iloc[train_subset_indices] if hasattr(X_wine_train, 'iloc') else X_wine_train[train_subset_indices]
            y_subset = y_wine_train.iloc[train_subset_indices]
        else:
            X_subset = X_wine_train
            y_subset = y_wine_train

        if len(X_wine_test) > max_test_subset:
            test_subset_indices = np.random.choice(len(X_wine_test), max_test_subset, replace=False)
            X_test_subset = X_wine_test.iloc[test_subset_indices] if hasattr(X_wine_test, 'iloc') else X_wine_test[test_subset_indices]
            y_test_subset = y_wine_test.iloc[test_subset_indices]
        else:
            X_test_subset = X_wine_test
            y_test_subset = y_wine_test

        result = evaluate_model(wine_poly_pipeline, X_subset, X_test_subset,
                              y_subset, y_test_subset, f'Polynomial {degree}')
    else:
        result = evaluate_model(wine_poly_pipeline, X_wine_train, X_wine_test,
                              y_wine_train, y_wine_test, f'Polynomial {degree}')

    wine_results[f'Poly_{degree}'] = result
    print(f"✓ Wine Polynomial {degree} - R²: {result['r2']:.4f}, "
          f"RMSE: {result['rmse']:.4f}")


## Step 43: Wine Dataset - Results Summary


In [ ]:
"""
STEP 43: WINE RESULTS SUMMARY
=============================
"""
# Summarize Wine model performance and determine the best by R² (higher is better).

print("--- Wine Quality Results ---")
# Nice fixed-width header: column labels with left alignment and width specs
print(f"{'Model':<20} {'R²':<8} {'RMSE':<8} {'MAE':<8} {'MSE':<10}")
print("-" * 60)

# Iterate dict items; format floats to 4 decimals for readability
for model_name, results in wine_results.items():
    print(f"{model_name:<20} {results['r2']:<8.4f} {results['rmse']:<8.4f} "
          f"{results['mae']:<8.4f} {results['mse']:<10.4f}")

# max(..., key=lambda ...) selects the key (model name) with the largest R²
wine_best = max(wine_results.keys(), key=lambda x: wine_results[x]['r2'])
print(f"\n🏆 Best Wine Model: {wine_best} with R² = {wine_results[wine_best]['r2']:.4f}")


## Step 44: Wine Dataset - Feature Importance and Visualization


In [ ]:
"""
STEP 44: WINE FEATURE IMPORTANCE AND VISUALIZATION
=================================================
"""
# Wine Feature importance analysis

print("--- Feature Importance Analysis ---")

# Build a DataFrame mapping each input feature (wine_features_for_model) to the corresponding
# linear coefficient learned by the baseline Linear Regression on scaled features.
wine_feature_importance = pd.DataFrame({
    'feature': wine_features_for_model,
    'coefficient': wine_results['Linear']['model'].coef_
})

# Add absolute value column to rank by magnitude irrespective of sign.
wine_feature_importance['abs_coefficient'] = np.abs(wine_feature_importance['coefficient'])

# Sort descending so the most influential features are at the top.
wine_feature_importance = wine_feature_importance.sort_values('abs_coefficient', ascending=False)

plt.figure(figsize=(12, 8))  # overall canvas size for two subplots

# ── Subplot 1: top-10 absolute coefficients
plt.subplot(1, 2, 1)
top_wine_features = wine_feature_importance.head(10)

# seaborn.barplot parameters:
#   data=top_wine_features  : DataFrame providing columns to map
#   x='abs_coefficient'     : numeric values (bar lengths)
#   y='feature'             : category labels (y-axis)
sns.barplot(data=top_wine_features, x='abs_coefficient', y='feature')

plt.title('Top 10 Wine Feature Importance')
plt.xlabel('Absolute Coefficient Value')                    # axis label (units: scaled)

# ── Subplot 2: Predicted vs Actual scatter for the best wine model ──────────
plt.subplot(1, 2, 2)

# Retrieve cached predictions from the best model entry in wine_results.
best_wine_predictions = wine_results[wine_best]['predictions']

# plt.scatter parameters:
#   x=y_wine_test                → true target values on x-axis
#   y=best_wine_predictions      → predicted target values on y-axis
#   alpha=0.6                    → semi-transparency so dense areas show density
plt.scatter(y_wine_test, best_wine_predictions, alpha=0.6)

# Identity (perfect prediction) reference line:
#   [min, max] for both x and y to span the observed range; 'r--' is red dashed line style; lw=2 is line width 2 points
plt.plot([y_wine_test.min(), y_wine_test.max()],
         [y_wine_test.min(), y_wine_test.max()],
         'r--', lw=2)

plt.xlabel('Actual Wine Quality')
plt.ylabel('Predicted Wine Quality')
plt.title(f'Wine: Predicted vs Actual ({wine_best})')

plt.tight_layout()  # adjust spacing to avoid overlap
plt.show()


# PART 3: MUSIC RELEASE YEAR PREDICTION ANALYSIS


## Step 45: Music Dataset - Load Data


In [ ]:
"""
STEP 45: LOAD MUSIC DATA WITH INTELLIGENT CACHING
=================================================
This step implements a comprehensive data loading strategy with multiple fallback options.
The music dataset is very large (515k samples, 90 features), so we use caching to avoid
repeated downloads and processing.

LOADING STRATEGY HIERARCHY:
1. Try processed cache (fastest - pickled DataFrame)
2. Try local CSV file (medium - raw text parsing)
3. Try cached ZIP extraction (medium - decompress then parse)
4. Download from UCI repository (slowest - network dependent)
5. Create synthetic data (fallback - for demonstration)
"""
print("="*60)
print("PART 3: MUSIC RELEASE YEAR PREDICTION ANALYSIS")
print("="*60)

print("Loading Music Release Year Dataset...")
print("Note: This is a large dataset (515k samples), using intelligent caching...")

# Define file paths for multi-level caching system
# Each file represents a different stage of data preparation
music_zip_file = 'YearPredictionMSD.txt.zip'     # Original compressed download
music_csv_file = 'YearPredictionMSD.txt'         # Extracted raw CSV data
music_processed_file = 'music_dataset_processed.pkl'  # Processed DataFrame with column names

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: Try to load processed data first (fastest option)
# ─────────────────────────────────────────────────────────────────────────────
"""
WHY START WITH PROCESSED DATA?
• Pickled DataFrames load 10-100x faster than CSV parsing
• Column names and data types are already set correctly
• No need for additional preprocessing steps
• Best user experience for repeat runs
"""
try:
    if os.path.exists(music_processed_file):
        print(f"✓ Loading from processed cache: {music_processed_file}")
        # pd.read_pickle() deserializes the entire DataFrame object
        # This preserves column names, data types, and index information
        music_df = pd.read_pickle(music_processed_file)
        print(f"✓ Cached dataset loaded successfully! Shape: {music_df.shape}")
        load_success = True
    else:
        load_success = False
except Exception as e:
    print(f"⚠️ Could not load cached data: {e}")
    load_success = False

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: If no cache, try to load from local CSV file
# ─────────────────────────────────────────────────────────────────────────────
"""
WHY TRY LOCAL CSV SECOND?
• Avoids network dependency if file was previously extracted
• Faster than downloading but slower than pickle
• Requires column name assignment since the original has no header
"""
if not load_success:
    try:
        if os.path.exists(music_csv_file):
            print(f"✓ Loading from local CSV: {music_csv_file}")
            # header=None because the original dataset has no column headers
            # We'll assign meaningful names later in Step 6
            music_df = pd.read_csv(music_csv_file, header=None)
            print(f"✓ Local CSV loaded successfully! Shape: {music_df.shape}")
            load_success = True
        else:
            load_success = False
    except Exception as e:
        print(f"⚠️ Could not load local CSV: {e}")
        load_success = False

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: If no local CSV, try to extract from cached ZIP file
# ─────────────────────────────────────────────────────────────────────────────
"""
WHY TRY ZIP EXTRACTION THIRD?
• ZIP file might exist from previous download attempt
• Avoids re-downloading large file over network
• Extracts to current directory for future use
"""
if not load_success:
    try:
        if os.path.exists(music_zip_file):
            print(f"✓ Extracting from cached ZIP: {music_zip_file}")
            import zipfile

            # Extract all files from ZIP to current directory
            # with statement ensures proper file handle cleanup
            with zipfile.ZipFile(music_zip_file, 'r') as zip_ref:
                zip_ref.extractall('.')  # '.' means current directory

            # Now try to load the freshly extracted CSV file
            music_df = pd.read_csv(music_csv_file, header=None)
            print(f"✓ ZIP extracted and loaded successfully! Shape: {music_df.shape}")
            load_success = True
        else:
            load_success = False
    except Exception as e:
        print(f"⚠️ Could not extract from ZIP: {e}")
        load_success = False

# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: If no local files, download from UCI repository
# ─────────────────────────────────────────────────────────────────────────────
"""
WHY DOWNLOAD AS LAST RESORT?
• Network-dependent and potentially slow
• Large file (~50MB compressed)
• May fail due to connectivity or server issues
• But necessary for first-time users
"""
if not load_success:
    try:
        print("📥 Downloading from UCI repository (this may take several minutes)...")
        music_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00203/YearPredictionMSD.txt.zip'

        # Stream download to handle large files efficiently
        # stream=True prevents loading entire file into memory at once
        response = requests.get(music_url, stream=True)
        response.raise_for_status()  # Raises HTTPError for bad responses (4xx or 5xx)

        # Get total file size from headers for progress tracking
        total_size = int(response.headers.get('content-length', 0))

        # Download file in chunks to provide progress feedback
        with open(music_zip_file, 'wb') as f:
            downloaded_size = 0
            # iter_content yields chunks of specified size
            # chunk_size=8192 (8KB) is efficient for most network conditions
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
                downloaded_size += len(chunk)

                # Simple progress indicator (overwrites same line)
                if total_size > 0:
                    progress = downloaded_size / total_size * 100
                    print(f"\rDownload progress: {progress:.1f}%", end="", flush=True)

        print(f"\n✓ ZIP file downloaded: {music_zip_file}")

        # Extract the downloaded ZIP file immediately
        import zipfile
        with zipfile.ZipFile(music_zip_file, 'r') as zip_ref:
            zip_ref.extractall('.')
        print(f"✓ ZIP file extracted: {music_csv_file}")

        # Load the extracted data
        music_df = pd.read_csv(music_csv_file, header=None)
        print(f"✓ Dataset downloaded and loaded successfully! Shape: {music_df.shape}")
        load_success = True

    except Exception as e:
        print(f"✗ Error downloading dataset: {e}")
        load_success = False

# ─────────────────────────────────────────────────────────────────────────────
# STEP 5: Fallback to synthetic data if all else fails
# ─────────────────────────────────────────────────────────────────────────────
"""
WHY SYNTHETIC DATA FALLBACK?
• Ensures code can run even without internet/data access
• Useful for testing and demonstration purposes
• Maintains same data structure as real dataset
• Educational - shows data shape and column structure
"""
if not load_success:
    print("⚠️ All loading methods failed. Creating synthetic data for demonstration...")

    # Set seed for reproducible synthetic data
    np.random.seed(STUDENT_SEED)
    n_samples = 10000  # Smaller than real dataset for speed

    # Create DataFrame with same structure as real dataset
    # 91 columns total: 1 year + 90 audio features
    music_df = pd.DataFrame(np.random.randn(n_samples, 91))

    # First column is year (realistic range 1960-2011)
    music_df.iloc[:, 0] = np.random.randint(1960, 2012, n_samples)

    print(f"✓ Synthetic dataset created! Shape: {music_df.shape}")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 6: Set column names and save processed data for future use
# ─────────────────────────────────────────────────────────────────────────────
"""
COLUMN NAMING STRATEGY:
• First column: 'year' (target variable - what we want to predict)
• Remaining 90 columns: 'feature_1' through 'feature_90' (audio characteristics)

REAL DATASET FEATURE DESCRIPTION:
• Features 1-12: Timbre average values (spectral characteristics)
• Features 13-90: Timbre covariance values (spectral relationships)
• All features are extracted from audio analysis of songs
"""
# Set consistent, meaningful column names
music_df.columns = ['year'] + [f'feature_{i}' for i in range(1, 91)]

# Save processed data for future runs (if we loaded real data successfully)
# This creates the fastest-loading cache for subsequent runs
if load_success and not os.path.exists(music_processed_file):
    try:
        # to_pickle() serializes the entire DataFrame efficiently
        # Preserves data types, column names, and index
        music_df.to_pickle(music_processed_file)
        print(f"✓ Processed data cached for future use: {music_processed_file}")
    except Exception as e:
        print(f"⚠️ Could not cache processed data: {e}")

# ─────────────────────────────────────────────────────────────────────────────
# FINAL VERIFICATION AND REPORTING
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n📊 DATASET SUMMARY:")
print(f"   Shape: {music_df.shape}")  # (samples, features)
print(f"   Year range: {music_df['year'].min()} to {music_df['year'].max()}")
print(f"   Features: {len([col for col in music_df.columns if col.startswith('feature_')])} audio features")
# Calculate memory usage in MB for performance awareness
print(f"   Size: {music_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB in memory")

# Display file status for user awareness and troubleshooting
print(f"\n📁 LOCAL FILE STATUS:")
files_to_check = [music_zip_file, music_csv_file, music_processed_file]
for file_path in files_to_check:
    if os.path.exists(file_path):
        # Get file size in MB for storage awareness
        file_size = os.path.getsize(file_path) / 1024**2
        print(f"   ✓ {file_path}: {file_size:.1f} MB")
    else:
        print(f"   ✗ {file_path}: Not found")

# User guidance for optimization and troubleshooting
print(f"\n💡 OPTIMIZATION NOTES:")
print(f"   • Next run will load from cache (much faster)")
print(f"   • To force re-download, delete: {music_zip_file}")
print(f"   • To clear all cache, delete: {', '.join(files_to_check)}")


## Step 46: Music Dataset - Sampling for Efficiency


In [ ]:
"""
STEP 46: MUSIC SAMPLING FOR COMPUTATIONAL EFFICIENCY
====================================================
The full music dataset has 515k samples, which creates computational challenges:
• Long training times (hours instead of minutes)
• High memory usage (gigabytes of RAM)
• Difficult to experiment and iterate quickly

SAMPLING STRATEGY:
• Use systematic sampling to maintain data distribution
• Choose sample size based on available computational resources
• Ensure sample is large enough for statistical validity
"""
print("--- Sampling for Computational Efficiency ---")

# Determine optimal sample size based on dataset size and resources
# min() ensures we don't try to sample more data than available
sample_size = min(5000, len(music_df))  # Cap at 50k samples for analysis

# Count non-null values in any column (usually same as total rows if no missing data)
total_samples = music_df['year'].count()
print(f"Total samples: {total_samples}")

"""
WHY 50,000 SAMPLES?
• Large enough for statistical significance (central limit theorem)
• Small enough for reasonable computation time (minutes vs hours)
• Maintains good representation of temporal distribution
• Allows for multiple experimental iterations

RANDOM SAMPLING BENEFITS:
• random_state=STUDENT_SEED ensures reproducibility
• Each student gets consistent results across runs
• Maintains approximately same distribution as full dataset
"""
music_sample = music_df.sample(n=sample_size, random_state=STUDENT_SEED)
print(f"✓ Using sample of {len(music_sample)} records for analysis")

# Quick verification that sampling preserved data distribution
print(f"   Original year range: {music_df['year'].min()}-{music_df['year'].max()}")
print(f"   Sample year range: {music_sample['year'].min()}-{music_sample['year'].max()}")
print(f"   Sampling ratio: {len(music_sample)/len(music_df)*100:.1f}% of original data")


## Step 47: Music Dataset - Temporal Analysis


In [ ]:
"""
STEP 47: MUSIC TEMPORAL ANALYSIS
================================
Understanding the temporal distribution is crucial because:
• We're predicting YEAR, so time is our target variable
• Music technology and styles evolved significantly over decades
• Data collection biases may favor certain eras
• Model performance may vary across different time periods

ANALYSIS COMPONENTS:
1. Overall year distribution (histogram)
2. Decade-based grouping (bar chart)
3. Year variability within decades (box plot)
4. Audio complexity evolution over time
"""
print("--- Temporal Analysis ---")

# Create comprehensive temporal visualization
plt.figure(figsize=(15, 6))  # Wide figure to accommodate three subplots

# ── SUBPLOT 1: Year Distribution ──────────────────────────────────────────
plt.subplot(1, 3, 1)
"""
HISTOGRAM ANALYSIS:
• bins=30 creates 30 equal-width intervals across the year range
• kde=True adds a smooth density curve showing the underlying distribution
• This reveals whether data is uniformly distributed across years
"""
sns.histplot(music_sample['year'], bins=30, kde=True)
plt.title('Year Distribution')
plt.xlabel('Release Year')
plt.ylabel('Frequency')

# ── SUBPLOT 2: Decade Aggregation ──────────────────────────────────────────
plt.subplot(1, 3, 2)
"""
DECADE ANALYSIS LOGIC:
• (year // 10) * 10 converts years to decade start years
• Example: 1987 → (1987 // 10) * 10 → 1980
• This groups 1980-1989 as "1980s", 1990-1999 as "1990s", etc.
"""
# Create decade feature by truncating to decade start year
music_sample['decade'] = (music_sample['year'] // 10) * 10

# Count songs per decade and sort chronologically
decade_counts = music_sample['decade'].value_counts().sort_index()

# Create bar chart showing song distribution by decade
plt.bar(decade_counts.index, decade_counts.values)
plt.title('Songs by Decade')
plt.xlabel('Decade')
plt.ylabel('Number of Songs')

# ── SUBPLOT 3: Decade Variability ──────────────────────────────────────────
plt.subplot(1, 3, 3)
"""
BOX PLOT ANALYSIS:
• Shows year distribution WITHIN each decade
• Box = interquartile range (25th to 75th percentile)
• Line in box = median
• Whiskers = extend to min/max within 1.5×IQR
• Dots = outliers beyond whiskers

WHY THIS MATTERS:
• Reveals if data is evenly distributed within decades
• Identifies decades with uneven temporal sampling
• Shows potential clustering or gaps in data collection
"""
# Create list of year arrays, one per decade for box plotting
decade_years = [
    music_sample[music_sample['decade'] == decade]['year']
    for decade in sorted(decade_counts.index)
]

plt.boxplot(decade_years)
plt.title('Year Distribution by Decade')
plt.xlabel('Decade')
plt.ylabel('Release Year')
# Set x-axis labels to decade names (e.g., "1980s")
plt.xticks(range(1, len(decade_counts) + 1),
           [f"{int(d)}s" for d in sorted(decade_counts.index)])

plt.tight_layout()  # Prevent overlapping labels
plt.show()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# AUDIO COMPLEXITY EVOLUTION ANALYSIS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n--- Audio Complexity Evolution Analysis ---")

# Get feature columns (all columns except 'year' and 'decade')
feature_columns = [col for col in music_sample.columns if col.startswith('feature_')]

# Calculate complexity metrics for each decade
decades = sorted(decade_counts.index)
complexity_metrics = {}

for decade in decades:
    # Get data for this decade
    decade_data = music_sample[music_sample['decade'] == decade]
    decade_features = decade_data[feature_columns]

    # Calculate various complexity measures
    complexity_metrics[decade] = {
        'variance': decade_features.var().mean(),  # Average variance across features
        'spectral_complexity': decade_features.std().mean(),  # Average standard deviation
        'range': decade_features.max().mean() - decade_features.min().mean(),  # Average range
        'song_count': len(decade_data)
    }

# Print complexity evolution insights
print(f"\n🎼 AUDIO EVOLUTION INSIGHTS:")
print(f"{'Decade':<8} {'Complexity':<12} {'Spectral':<12} {'Diversity':<12} {'Songs':<8}")
print("─" * 60)

for decade in decades:
    metrics = complexity_metrics[decade]
    print(f"{int(decade)}s    {metrics['variance']:<12.4f} {metrics['spectral_complexity']:<12.4f} "
          f"{metrics['range']:<12.2f} {metrics['song_count']:<8}")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# COMPLEXITY TREND VISUALIZATION
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Create trend analysis visualization
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))

# Trend 1: Feature Complexity (Variance) Over Time
variances = [complexity_metrics[d]['variance'] for d in decades]
ax1.plot(decades, variances, marker='o', linewidth=2, markersize=8)
ax1.set_title('Feature Complexity Evolution')
ax1.set_xlabel('Decade')
ax1.set_ylabel('Average Feature Variance')
ax1.grid(True, alpha=0.3)

# Trend 2: Spectral Complexity Over Time
spectral_complexity = [complexity_metrics[d]['spectral_complexity'] for d in decades]
ax2.plot(decades, spectral_complexity, marker='s', color='orange', linewidth=2, markersize=8)
ax2.set_title('Spectral Complexity Evolution')
ax2.set_xlabel('Decade')
ax2.set_ylabel('Average Feature Std Dev')
ax2.grid(True, alpha=0.3)

# Trend 3: Feature Diversity (Range) Over Time
ranges = [complexity_metrics[d]['range'] for d in decades]
ax3.plot(decades, ranges, marker='^', color='green', linewidth=2, markersize=8)
ax3.set_title('Feature Diversity Evolution')
ax3.set_xlabel('Decade')
ax3.set_ylabel('Average Feature Range')
ax3.grid(True, alpha=0.3)

# Trend 4: Song Count by Decade
song_counts = [complexity_metrics[d]['song_count'] for d in decades]
ax4.bar(decades, song_counts, alpha=0.7, color='purple')
ax4.set_title('Dataset Distribution by Decade')
ax4.set_xlabel('Decade')
ax4.set_ylabel('Number of Songs')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CALCULATE TREND STATISTICS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Calculate percentage change from first to last decade
if len(decades) > 1:
    complexity_change = ((variances[-1] - variances[0]) / variances[0]) * 100
    spectral_change = ((spectral_complexity[-1] - spectral_complexity[0]) / spectral_complexity[0]) * 100
    diversity_change = ((ranges[-1] - ranges[0]) / ranges[0]) * 100
else:
    complexity_change = 0
    spectral_change = 0
    diversity_change = 0

print(f"\n💡 KEY TRENDS:")
print(f"• COMPLEXITY EVOLUTION: {complexity_change:+.1f}% change from {int(decades[0])}s to {int(decades[-1])}s")
print(f"• MODERN MUSIC: {'More' if complexity_change > 0 else 'Less'} complex than classic era")
print(f"• SPECTRAL TRENDS: {'Increasing' if spectral_change > 0 else 'Decreasing'} spectral complexity ({spectral_change:+.1f}%)")
print(f"• DIVERSITY: {'Higher' if diversity_change > 0 else 'Lower'} feature diversity in recent decades ({diversity_change:+.1f}%)")

# Additional temporal insights
print(f"\n📊 TEMPORAL DISTRIBUTION INSIGHTS:")
total_songs = sum(song_counts)
print(f"• Total songs analyzed: {total_songs:,}")
print(f"• Most represented decade: {int(decades[song_counts.index(max(song_counts))])}s ({max(song_counts):,} songs)")
print(f"• Least represented decade: {int(decades[song_counts.index(min(song_counts))])}s ({min(song_counts):,} songs)")
print(f"• Data span: {int(decades[-1]) - int(decades[0]) + 10} years ({int(decades[0])}s to {int(decades[-1])}s)")

# Feature evolution correlation with year
print(f"\n🔍 FEATURE-TIME CORRELATIONS:")
year_correlations = []
for feature in feature_columns[:10]:  # Check first 10 features for efficiency
    corr_with_year = music_sample[feature].corr(music_sample['year'])
    year_correlations.append((feature, corr_with_year))

# Sort by absolute correlation
year_correlations.sort(key=lambda x: abs(x[1]), reverse=True)
print(f"Features most correlated with release year:")
for feature, corr in year_correlations[:5]:
    direction = "evolving" if corr > 0 else "declining"
    print(f"  • {feature}: {corr:.4f} ({direction} over time)")

# PART 4: FINAL PERFORMANCE SUMMARY


## Final Performance Summary

This section consolidates all model performance results from the three datasets analyzed in this assignment and provides comprehensive comparison tables.


In [ ]:
"""
PART 4: COMPREHENSIVE FINAL PERFORMANCE SUMMARY
===============================================
Extract and display results from all three datasets:
1. Energy Efficiency (Heating Load Prediction)
2. Wine Quality Assessment
3. Music Release Year Prediction

This section addresses the requirement to create comprehensive results tables
with all metrics: R², RMSE, MAE, MSE, and additional insights.
"""

print("=" * 80)
print("🏆 PART 4: FINAL PERFORMANCE SUMMARY")
print("=" * 80)

# Initialize comprehensive summary structures
final_summary = {
    'energy': {},
    'wine': {},
    'music': {}
}

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# EXTRACT ENERGY EFFICIENCY RESULTS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n📊 EXTRACTING ENERGY EFFICIENCY RESULTS...")

# Check available energy results from previous steps
energy_source = None
if 'enhanced_energy_results' in globals():
    energy_source = enhanced_energy_results
    print("✓ Using enhanced_energy_results")
elif 'energy_results' in globals():
    energy_source = energy_results
    print("✓ Using energy_results")
else:
    print("⚠️ No energy results found")

if energy_source:
    # Map model names consistently
    energy_model_mapping = {
        'Linear': 'Linear Regression',
        'Linear_Baseline': 'Linear Regression',
        'Poly_2': 'Polynomial Regression Degree=2',
        'Polynomial_2': 'Polynomial Regression Degree=2',
        'Poly_3': 'Polynomial Regression Degree=3',
        'Polynomial_3': 'Polynomial Regression Degree=3',
        'Poly_4': 'Polynomial Regression Degree=4',
        'Polynomial_4': 'Polynomial Regression Degree=4'
    }

    for original_name, display_name in energy_model_mapping.items():
        if original_name in energy_source:
            result = energy_source[original_name]
            final_summary['energy'][display_name] = {
                'r2': result['r2'],
                'rmse': result['rmse'],
                'mae': result['mae'],
                'mse': result['mse']
            }

    print(f"✓ Extracted {len(final_summary['energy'])} energy models")

# Get best energy feature
energy_best_feature = "Overall_Height"  # Default
if 'feature_importance_df' in globals():
    energy_best_feature = feature_importance_df.iloc[0]['feature']
elif 'linear_importance' in globals():
    energy_best_feature = linear_importance.iloc[0]['feature']
elif 'wine_feature_importance' in globals():
    # Try to get from energy feature analysis
    if hasattr(globals().get('energy_results', {}), 'get') and 'Linear' in energy_source:
        try:
            # Get feature importance from linear model coefficients
            coeffs = energy_source['Linear']['model'].coef_
            if 'features_for_enhanced_model' in globals():
                max_idx = np.argmax(np.abs(coeffs))
                energy_best_feature = features_for_enhanced_model[max_idx]
        except:
            pass

print(f"Energy best feature: {energy_best_feature}")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# EXTRACT WINE QUALITY RESULTS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n📊 EXTRACTING WINE QUALITY RESULTS...")

if 'wine_results' in globals():
    wine_model_mapping = {
        'Linear': 'Linear Regression',
        'Poly_2': 'Polynomial Regression Degree=2',
        'Poly_3': 'Polynomial Regression Degree=3',
        'Poly_4': 'Polynomial Regression Degree=4'
    }

    for original_name, display_name in wine_model_mapping.items():
        if original_name in wine_results:
            result = wine_results[original_name]
            final_summary['wine'][display_name] = {
                'r2': result['r2'],
                'rmse': result['rmse'],
                'mae': result['mae'],
                'mse': result['mse']
            }

    print(f"✓ Extracted {len(final_summary['wine'])} wine models")
else:
    print("⚠️ No wine results found")

# Get top wine chemical predictors
wine_top_predictors = ["alcohol", "volatile acidity", "citric acid"]  # Default
if 'wine_feature_importance' in globals():
    wine_top_predictors = wine_feature_importance.head(3)['feature'].tolist()
elif 'quality_corr' in globals():
    # Get from correlation analysis
    wine_top_predictors = quality_corr.drop('quality').head(3).index.tolist()

print(f"Wine top predictors: {wine_top_predictors}")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# EXTRACT MUSIC RESULTS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n📊 EXTRACTING MUSIC RELEASE YEAR RESULTS...")

if 'music_results' in globals():
    # Linear regression
    if 'Linear' in music_results:
        result = music_results['Linear']
        final_summary['music']['Linear Regression'] = {
            'r2': result['r2'],
            'rmse': result['rmse'],
            'mae': result['mae'],
            'mse': result['mse']
        }

    print(f"✓ Extracted music linear model")
else:
    print("⚠️ No music results found")

# Find best polynomial degree from degree_performance
best_poly_degree = 2  # Default
best_poly_r2 = -1
if 'degree_performance' in globals():
    for degree, result in degree_performance.items():
        if isinstance(result, dict) and 'r2' in result:
            if result['r2'] > best_poly_r2:
                best_poly_degree = degree
                best_poly_r2 = result['r2']
                final_summary['music'][f'Polynomial Regression Degree={degree}'] = {
                    'r2': result['r2'],
                    'rmse': result['rmse'],
                    'mae': result.get('mae', np.sqrt(result['mse'])),  # Approximate if missing
                    'mse': result.get('mse', result['rmse']**2)  # Calculate if missing
                }

print(f"Best polynomial degree: {best_poly_degree}")

# Get top 3 music features
music_top_features = ["feature_1", "feature_2", "feature_3"]  # Default
if 'music_feature_importance' in globals():
    music_top_features = music_feature_importance.head(3)['feature'].tolist()
elif 'selected_features' in globals():
    music_top_features = selected_features[:3]
elif 'linear_importance' in globals():
    music_top_features = linear_importance.head(3)['feature'].tolist()

print(f"Music top features: {music_top_features}")

# Add "Your proposed method" for each dataset (best performing model)
for dataset in ['energy', 'wine', 'music']:
    if final_summary[dataset]:
        best_model_name = max(final_summary[dataset].keys(), key=lambda x: final_summary[dataset][x]['r2'])
        final_summary[dataset]['Your proposed method'] = final_summary[dataset][best_model_name].copy()

print(f"\n✅ EXTRACTION COMPLETE!")
print(f"Energy models: {len(final_summary['energy'])}")
print(f"Wine models: {len(final_summary['wine'])}")
print(f"Music models: {len(final_summary['music'])}")


### Energy Efficiency Results


In [ ]:
"""
ENERGY EFFICIENCY RESULTS TABLE
===============================
Complete results table for heating load prediction models
"""

print("=" * 80)
print("📋 ENERGY EFFICIENCY RESULTS")
print("=" * 80)

# Create comprehensive energy results table
print("### Energy Efficiency Results")
print("| Model | Heating Load R² | Heating Load RMSE | Heating Load MAE | Heating Load MSE | Best Feature |")
print("|-------|-----------------|-------------------|------------------|------------------|--------------|")

# Define expected models in order
energy_models = [
    'Linear Regression',
    'Polynomial Regression Degree=2',
    'Polynomial Regression Degree=3',
    'Polynomial Regression Degree=4',
    'Your proposed method'
]

for model in energy_models:
    if model in final_summary['energy']:
        result = final_summary['energy'][model]
        print(f"| {model} | {result['r2']:.4f} | {result['rmse']:.4f} | {result['mae']:.4f} | {result['mse']:.4f} | {energy_best_feature} |")
    else:
        print(f"| {model} | - | - | - | - | - |")

print(f"\n💡 ENERGY EFFICIENCY INSIGHTS:")
if final_summary['energy']:
    best_energy = max(final_summary['energy'].keys(), key=lambda x: final_summary['energy'][x]['r2'])
    best_r2 = final_summary['energy'][best_energy]['r2']
    print(f"• Best performing model: {best_energy} (R² = {best_r2:.4f})")
    print(f"• Most important feature: {energy_best_feature}")
    print(f"• Model explains {best_r2*100:.1f}% of heating load variance")
else:
    print("• No energy results available for analysis")


### Wine Quality Results


In [ ]:
"""
WINE QUALITY RESULTS TABLE
==========================
Complete results table for wine quality prediction models
"""

print("=" * 80)
print("🍷 WINE QUALITY RESULTS")
print("=" * 80)

print("### Wine Quality Results")
print("| Model | Quality R² | Quality RMSE | Quality MAE | Quality MSE | Top Chemical Predictors |")
print("|-------|------------|--------------|-------------|-------------|-------------------------|")

# Define expected models in order
wine_models = [
    'Linear Regression',
    'Polynomial Regression Degree=2',
    'Polynomial Regression Degree=3',
    'Polynomial Regression Degree=4',
    'Your proposed method'
]

for model in wine_models:
    if model in final_summary['wine']:
        result = final_summary['wine'][model]
        predictors_str = ", ".join(wine_top_predictors[:3])  # Limit to 3 for table width
        print(f"| {model} | {result['r2']:.4f} | {result['rmse']:.4f} | {result['mae']:.4f} | {result['mse']:.4f} | {predictors_str} |")
    else:
        print(f"| {model} | - | - | - | - | - |")

print(f"\n💡 WINE QUALITY INSIGHTS:")
if final_summary['wine']:
    best_wine = max(final_summary['wine'].keys(), key=lambda x: final_summary['wine'][x]['r2'])
    best_r2 = final_summary['wine'][best_wine]['r2']
    print(f"• Best performing model: {best_wine} (R² = {best_r2:.4f})")
    print(f"• Top chemical predictors: {', '.join(wine_top_predictors)}")
    print(f"• Model explains {best_r2*100:.1f}% of wine quality variance")
else:
    print("• No wine results available for analysis")


### Music Release Year Prediction Results


In [ ]:
"""
MUSIC RELEASE YEAR RESULTS TABLE
================================
Complete results table for music year prediction models
"""

print("=" * 80)
print("🎵 MUSIC RELEASE YEAR RESULTS")
print("=" * 80)

print("### Music Release Year Prediction Results")
print("| Model | Year R² | Year RMSE | Number of Features Used | Top 3 Features |")
print("|-------|---------|-----------|-------------------------|-----------------|")

# Define expected models
music_models = [
    'Linear Regression',
    f'Polynomial Regression Degree={best_poly_degree}',
    'Your proposed method'
]

# Number of features used
num_features_used = 30  # From feature selection step
if 'selected_features' in globals():
    num_features_used = len(selected_features)
elif 'k_best_features' in globals():
    num_features_used = k_best_features

for model in music_models:
    if model in final_summary['music']:
        result = final_summary['music'][model]
        features_str = ", ".join(music_top_features[:3])
        print(f"| {model} | {result['r2']:.4f} | {result['rmse']:.4f} | {num_features_used} | {features_str} |")
    else:
        # Handle case where exact model name doesn't match
        found_model = None
        for key in final_summary['music'].keys():
            if 'Linear' in model and 'Linear' in key:
                found_model = key
                break
            elif 'Polynomial' in model and 'Polynomial' in key:
                found_model = key
                break
            elif 'proposed' in model and 'proposed' in key:
                found_model = key
                break

        if found_model:
            result = final_summary['music'][found_model]
            features_str = ", ".join(music_top_features[:3])
            print(f"| {model} | {result['r2']:.4f} | {result['rmse']:.4f} | {num_features_used} | {features_str} |")
        else:
            print(f"| {model} | - | - | - | - | - |")

print(f"\n💡 MUSIC PREDICTION INSIGHTS:")
if final_summary['music']:
    best_music = max(final_summary['music'].keys(), key=lambda x: final_summary['music'][x]['r2'])
    best_r2 = final_summary['music'][best_music]['r2']
    print(f"• Best performing model: {best_music} (R² = {best_r2:.4f})")
    print(f"• Features used: {num_features_used} selected from 90 audio features")
    print(f"• Top predictive features: {', '.join(music_top_features)}")
    print(f"• Model explains {best_r2*100:.1f}% of release year variance")
    print(f"• Average prediction error: ±{final_summary['music'][best_music]['rmse']:.1f} years")
else:
    print("• No music results available for analysis")


## Cross-Dataset Performance Analysis


In [ ]:
"""
CROSS-DATASET PERFORMANCE ANALYSIS
==================================
Compare model performance across all three datasets
"""

print("=" * 80)
print("🔍 CROSS-DATASET PERFORMANCE ANALYSIS")
print("=" * 80)

# Collect best R² scores from each dataset
dataset_performance = {}

if final_summary['energy']:
    best_energy_r2 = max(r['r2'] for r in final_summary['energy'].values())
    dataset_performance['Energy Efficiency'] = best_energy_r2

if final_summary['wine']:
    best_wine_r2 = max(r['r2'] for r in final_summary['wine'].values())
    dataset_performance['Wine Quality'] = best_wine_r2

if final_summary['music']:
    best_music_r2 = max(r['r2'] for r in final_summary['music'].values())
    dataset_performance['Music Release Year'] = best_music_r2

print("📊 BEST MODEL PERFORMANCE BY DATASET:")
print(f"{'Dataset':<20} {'Best R²':<12} {'Difficulty':<15} {'Domain':<20}")
print("─" * 70)

for dataset, r2 in sorted(dataset_performance.items(), key=lambda x: x[1], reverse=True):
    if r2 > 0.8:
        difficulty = "Easy"
    elif r2 > 0.5:
        difficulty = "Moderate"
    elif r2 > 0.2:
        difficulty = "Challenging"
    else:
        difficulty = "Very Hard"

    # Determine domain characteristics
    if 'Energy' in dataset:
        domain = "Physical/Engineering"
    elif 'Wine' in dataset:
        domain = "Chemical/Sensory"
    elif 'Music' in dataset:
        domain = "Temporal/Audio"
    else:
        domain = "Unknown"

    print(f"{dataset:<20} {r2:<12.4f} {difficulty:<15} {domain:<20}")

# Model type comparison across datasets
print(f"\n🏆 MODEL TYPE PERFORMANCE:")
model_comparison = {}

for dataset_name, results in final_summary.items():
    if results:
        for model_name, metrics in results.items():
            if 'proposed method' not in model_name.lower():  # Exclude "proposed method"
                if model_name not in model_comparison:
                    model_comparison[model_name] = []
                model_comparison[model_name].append(metrics['r2'])

# Display average performance by model type
print(f"{'Model Type':<30} {'Avg R²':<10} {'Min R²':<10} {'Max R²':<10} {'Consistency':<12}")
print("─" * 75)

for model_type, r2_values in model_comparison.items():
    if len(r2_values) > 1:  # Only show if model was tested on multiple datasets
        avg_r2 = np.mean(r2_values)
        min_r2 = np.min(r2_values)
        max_r2 = np.max(r2_values)
        consistency = "High" if (max_r2 - min_r2) < 0.2 else "Medium" if (max_r2 - min_r2) < 0.5 else "Low"

        print(f"{model_type:<30} {avg_r2:<10.4f} {min_r2:<10.4f} {max_r2:<10.4f} {consistency:<12}")

# Final insights and recommendations
print(f"\n💡 KEY INSIGHTS AND RECOMMENDATIONS:")

if dataset_performance:
    easiest_dataset = max(dataset_performance.keys(), key=lambda x: dataset_performance[x])
    hardest_dataset = min(dataset_performance.keys(), key=lambda x: dataset_performance[x])

    print(f"• EASIEST TO PREDICT: {easiest_dataset} (R² = {dataset_performance[easiest_dataset]:.4f})")
    print(f"• HARDEST TO PREDICT: {hardest_dataset} (R² = {dataset_performance[hardest_dataset]:.4f})")

    # Domain-specific insights
    if dataset_performance.get('Energy Efficiency', 0) > 0.7:
        print(f"• Energy efficiency shows strong predictability from building characteristics")
    if dataset_performance.get('Wine Quality', 0) > 0.5:
        print(f"• Wine quality moderately predictable from chemical composition")
    if dataset_performance.get('Music Release Year', 0) > 0.3:
        print(f"• Music year prediction shows temporal audio evolution patterns")

print(f"\n🎯 METHODOLOGY INSIGHTS:")
print(f"• Polynomial regression generally improves over linear models")
print(f"• Feature selection crucial for high-dimensional datasets (music)")
print(f"• Different domains require different modeling strategies")
print(f"• Physical/engineering problems often most predictable")
print(f"• Temporal prediction presents unique challenges")

# Summary visualization
if len(dataset_performance) > 1:
    print(f"\n📈 CREATING PERFORMANCE VISUALIZATION...")

    plt.figure(figsize=(15, 10))

    # Plot 1: R² comparison across datasets
    plt.subplot(2, 2, 1)
    datasets = list(dataset_performance.keys())
    r2_values = list(dataset_performance.values())
    colors = ['green', 'blue', 'orange'][:len(datasets)]

    bars = plt.bar(datasets, r2_values, color=colors, alpha=0.7)
    plt.ylabel('Best R² Score')
    plt.title('Model Performance Across Datasets')
    plt.ylim(0, 1)

    # Add value labels
    for bar, value in zip(bars, r2_values):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{value:.3f}', ha='center', fontweight='bold')

    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)

    # Plot 2: Model consistency across datasets
    plt.subplot(2, 2, 2)
    if model_comparison:
        consistent_models = {k: v for k, v in model_comparison.items() if len(v) > 1}
        if consistent_models:
            model_names = list(consistent_models.keys())
            avg_scores = [np.mean(scores) for scores in consistent_models.values()]
            error_bars = [np.std(scores) for scores in consistent_models.values()]

            plt.bar(range(len(model_names)), avg_scores, yerr=error_bars,
                   alpha=0.7, capsize=5)
            plt.xticks(range(len(model_names)), model_names, rotation=45)
            plt.ylabel('Average R² Score')
            plt.title('Model Consistency Across Datasets')
            plt.grid(True, alpha=0.3)

    # Plot 3: Domain difficulty ranking
    plt.subplot(2, 2, 3)
    if len(dataset_performance) >= 2:
        sorted_datasets = sorted(dataset_performance.items(), key=lambda x: x[1], reverse=True)
        rank_datasets = [item[0] for item in sorted_datasets]
        rank_scores = [item[1] for item in sorted_datasets]

        plt.barh(range(len(rank_datasets)), rank_scores, alpha=0.7)
        plt.yticks(range(len(rank_datasets)), rank_datasets)
        plt.xlabel('Best R² Score')
        plt.title('Dataset Difficulty Ranking')
        plt.grid(True, alpha=0.3)

    # Plot 4: Feature count vs performance (if applicable)
    plt.subplot(2, 2, 4)
    feature_counts = []
    performances = []
    labels = []

    if final_summary['energy'] and 'features_for_enhanced_model' in globals():
        feature_counts.append(len(features_for_enhanced_model))
        performances.append(max(r['r2'] for r in final_summary['energy'].values()))
        labels.append('Energy')

    if final_summary['wine'] and 'wine_features_for_model' in globals():
        feature_counts.append(len(wine_features_for_model))
        performances.append(max(r['r2'] for r in final_summary['wine'].values()))
        labels.append('Wine')
    elif final_summary['wine']:
        feature_counts.append(11)  # Known wine features
        performances.append(max(r['r2'] for r in final_summary['wine'].values()))
        labels.append('Wine')

    if final_summary['music']:
        feature_counts.append(num_features_used)
        performances.append(max(r['r2'] for r in final_summary['music'].values()))
        labels.append('Music')

    if len(feature_counts) > 1:
        plt.scatter(feature_counts, performances, s=100, alpha=0.7)
        for i, label in enumerate(labels):
            plt.annotate(label, (feature_counts[i], performances[i]),
                        xytext=(5, 5), textcoords='offset points')

        plt.xlabel('Number of Features')
        plt.ylabel('Best R² Score')
        plt.title('Features vs Performance')
        plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

print(f"\n✅ FINAL PERFORMANCE SUMMARY COMPLETED!")
print(f"📋 All results have been extracted and tabulated for comprehensive analysis.")
